### Notebook 4: Tariff Pricing Agent

**Purpose:** Translate demand forecasts from the Demand Prediction Agent into optimal dynamic tariffs.

**Pricing Rules:**
| Utilization | Tariff Action | Multiplier |
|---|---|---|
| > 80% (Congested) | Surge pricing | ×1.5 → ₹22.5/kWh |
| 30–80% (Normal) | Baseline | ×1.0 → ₹15/kWh |
| < 30% (Off-peak) | Discount pricing | ×0.7 → ₹10.5/kWh |

**Evaluation Metrics:**
- Revenue Gain % vs fixed ₹15/kWh baseline
- Charger Utilization Rate before and after dynamic pricing
- Off-Peak Uplift % (simulated demand shift)

**Assumption:** Demand elasticity = −0.3 (standard short-run EV literature value). A 10% price increase → 3% demand reduction. Applied as simulation only; real elasticity requires A/B test data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 130

BASE  = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
PROC  = os.path.join(BASE, "data", "processed") + os.sep
PLOTS = os.path.join(BASE, "plots")             + os.sep
OUT   = os.path.join(BASE, "outputs")           + os.sep

preds = pd.read_csv(OUT + "demand_predictions.csv", parse_dates=['datetime'])
acn   = pd.read_csv(PROC + "acn_processed.csv",     parse_dates=['connection_time'])

print(f"Predictions : {preds.shape}")
print(f"ACN sessions: {acn.shape}")
preds.head(3)

### 4.1 Dynamic Tariff Rule Engine

In [ ]:
# Tariff parameters
FIXED_TARIFF       = 15.0   # ₹/kWh baseline
SURGE_MULT         = 1.50   # 50% premium when congested
DISCOUNT_MULT      = 0.70   # 30% discount when off-peak
DEMAND_ELASTICITY  = -0.30  # short-run elasticity (literature)

def compute_dynamic_tariff(util, base=FIXED_TARIFF,
                            surge=SURGE_MULT, discount=DISCOUNT_MULT):
    return np.where(util > 0.8, base * surge,
           np.where(util < 0.3, base * discount, base))

preds['dynamic_tariff'] = compute_dynamic_tariff(preds['pred_utilization'])
preds['tariff_regime']  = np.where(preds['pred_utilization'] > 0.8, 'Surge',
                          np.where(preds['pred_utilization'] < 0.3, 'Discount', 'Baseline'))

print("Tariff regime distribution:")
print(preds['tariff_regime'].value_counts())
print(f"\nAvg dynamic tariff : ₹{preds['dynamic_tariff'].mean():.2f}/kWh")
print(f"Fixed baseline     : ₹{FIXED_TARIFF:.2f}/kWh")

### 4.2 Revenue Simulation — Fixed vs Dynamic Tariff

In [ ]:
# Revenue = utilization × tariff (proxy; actual kWh data used for ACN)
preds['revenue_fixed']   = preds['utilization'] * FIXED_TARIFF

# Elasticity-adjusted demand after price change
preds['price_change_pct']      = (preds['dynamic_tariff'] - FIXED_TARIFF) / FIXED_TARIFF
preds['demand_adjustment']     = 1 + (DEMAND_ELASTICITY * preds['price_change_pct'])
preds['adjusted_utilization']  = (preds['utilization'] * preds['demand_adjustment']).clip(0, 1)
preds['revenue_dynamic_adj']   = preds['adjusted_utilization'] * preds['dynamic_tariff']

total_rev_fixed   = preds['revenue_fixed'].sum()
total_rev_dynamic = preds['revenue_dynamic_adj'].sum()
revenue_gain_pct  = ((total_rev_dynamic - total_rev_fixed) / total_rev_fixed) * 100

print(f"Total Revenue (Fixed)   : {total_rev_fixed:,.1f}")
print(f"Total Revenue (Dynamic) : {total_rev_dynamic:,.1f}")
print(f"Revenue Gain %          : {revenue_gain_pct:.2f}%")

### 4.3 Charger Utilization & Off-Peak Uplift

In [ ]:
util_before = preds['utilization'].mean()
util_after  = preds['adjusted_utilization'].mean()

# Off-peak uplift: discount drives more sessions to low-demand windows
offpeak_before = (preds['utilization'] < 0.3).sum()
offpeak_after  = int(offpeak_before * 1.15)  # 15% simulated shift
offpeak_uplift = ((offpeak_after - offpeak_before) / offpeak_before) * 100

print("=" * 50)
print("  Tariff Pricing Agent — Key Metrics")
print("=" * 50)
print(f"  Revenue Gain %              : {revenue_gain_pct:.2f}%")
print(f"  Charger Util Before         : {util_before:.2%}")
print(f"  Charger Util After          : {util_after:.2%}")
print(f"  Off-Peak Uplift (simulated) : {offpeak_uplift:.1f}%")

In [ ]:
regime_stats = preds.groupby('tariff_regime').agg(
    count               = ('utilization',        'count'),
    avg_utilization     = ('utilization',         'mean'),
    avg_dynamic_tariff  = ('dynamic_tariff',      'mean'),
    revenue_fixed_sum   = ('revenue_fixed',       'sum'),
    revenue_dynamic_sum = ('revenue_dynamic_adj', 'sum')
).reset_index()

regime_stats['revenue_gain_pct'] = (
    (regime_stats['revenue_dynamic_sum'] - regime_stats['revenue_fixed_sum'])
    / regime_stats['revenue_fixed_sum'] * 100
)
regime_stats.to_csv(OUT + "tariff_regime_stats.csv", index=False)
print(regime_stats.to_string(index=False))

### 4.4 Pricing Outcome Visualizations

In [ ]:
preds['hour'] = preds['datetime'].dt.hour

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Tariff distribution
axes[0,0].hist(preds['dynamic_tariff'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].axvline(FIXED_TARIFF, color='red', ls='--', lw=1.5, label=f'Fixed ₹{FIXED_TARIFF}')
axes[0,0].set_title('Dynamic Tariff Distribution', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Tariff (₹/kWh)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()

# 2. Revenue by regime
regime_stats.plot(kind='bar', x='tariff_regime',
                  y=['revenue_fixed_sum','revenue_dynamic_sum'],
                  ax=axes[0,1], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0,1].set_title('Revenue: Fixed vs Dynamic by Regime', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Tariff Regime')
axes[0,1].set_ylabel('Revenue (proxy units)')
axes[0,1].tick_params(axis='x', rotation=0)
axes[0,1].legend(['Fixed','Dynamic'])

# 3. Utilization before vs after
pd.DataFrame({
    'Scenario'   : ['Before (Fixed)', 'After (Dynamic)'],
    'Utilization': [util_before, util_after]
}).plot(kind='bar', x='Scenario', y='Utilization', ax=axes[1,0],
        color=['#e74c3c','#2ecc71'], edgecolor='white', legend=False)
axes[1,0].set_title('Charger Utilization: Before vs After', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('Mean Utilization Rate')
axes[1,0].tick_params(axis='x', rotation=0)
axes[1,0].axhline(0.8, color='red',   ls='--', lw=1, label='Congestion (80%)')
axes[1,0].axhline(0.3, color='green', ls='--', lw=1, label='Off-peak (30%)')
axes[1,0].legend(fontsize=8)

# 4. Avg tariff by hour
preds.groupby('hour')['dynamic_tariff'].mean().plot(
    kind='bar', ax=axes[1,1], color='coral', edgecolor='white')
axes[1,1].axhline(FIXED_TARIFF, color='navy', ls='--', lw=1.5,
                  label=f'Fixed ₹{FIXED_TARIFF}')
axes[1,1].set_title('Average Dynamic Tariff by Hour of Day', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Hour of Day')
axes[1,1].set_ylabel('Avg Tariff (₹/kWh)')
axes[1,1].legend()

plt.suptitle('Tariff Pricing Agent — Outcomes Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PLOTS + "tariff_pricing_outcomes.png", bbox_inches='tight')
plt.show()
print("📁 Saved: tariff_pricing_outcomes.png")

In [ ]:
# Save all pricing outputs
preds[[
    'datetime','station_id','utilization','pred_utilization',
    'pred_congestion_prob','dynamic_tariff','tariff_regime',
    'revenue_fixed','revenue_dynamic_adj','adjusted_utilization'
]].to_csv(OUT + "tariff_decisions.csv", index=False)

pricing_metrics = pd.DataFrame({
    'Metric': [
        'Fixed Baseline Tariff',
        'Surge Tariff (>80% util)',
        'Discount Tariff (<30% util)',
        'Revenue Gain %',
        'Charger Utilization Before',
        'Charger Utilization After',
        'Off-Peak Uplift % (simulated)',
        'Surge Regime % of timesteps',
        'Discount Regime % of timesteps',
        'Baseline Regime % of timesteps'
    ],
    'Value': [
        f"₹{FIXED_TARIFF}/kWh",
        f"₹{FIXED_TARIFF * SURGE_MULT}/kWh",
        f"₹{FIXED_TARIFF * DISCOUNT_MULT}/kWh",
        f"{revenue_gain_pct:.2f}%",
        f"{util_before:.2%}",
        f"{util_after:.2%}",
        f"{offpeak_uplift:.1f}%",
        f"{(preds['tariff_regime']=='Surge').mean()*100:.1f}%",
        f"{(preds['tariff_regime']=='Discount').mean()*100:.1f}%",
        f"{(preds['tariff_regime']=='Baseline').mean()*100:.1f}%"
    ]
})
pricing_metrics.to_csv(OUT + "pricing_metrics.csv", index=False)
print(pricing_metrics.to_string(index=False))
print("\n✅ Notebook 4 complete! Run Notebook 5 next.")